# 02 · Ön İşleme (Preprocessing)

1. Görsellerin güvenli yüklenmesi ve RGB'ye çevrilmesi
2. Grafik (heat-map) alanının kırpılması — eksenler/colorbar atılır, Range-Doppler hücrelerine dokunulmaz
3. En-boy oranını koruyan yeniden boyutlandırma (512 × 128 değerlendirmesi dahil)
4. Kayıt bazlı train/val/test bölmesi → `outputs/splits.csv`
5. Yalnızca train bölümünden normalizasyon istatistikleri → `outputs/normalization.json`
6. Tensör dönüşümü

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "passive_radar").is_dir())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from passive_radar.config import CLASS_NAMES, ProjectConfig
from passive_radar.data.indexing import (
    SPLIT_NAMES, assert_no_group_leakage, deduplicate_records, records_in_split,
    save_split_csv, scan_labeled_directory, split_by_source,
)
from passive_radar.data.preprocessing import (
    compute_channel_stats, crop_plot_area, letterbox_resize, load_rgb_image,
    preprocess_image, save_channel_stats,
)
from passive_radar.data.transforms import array_to_tensor
from passive_radar.evaluation.plots import plot_image_grid

config = ProjectConfig.default(PROJECT_ROOT)
pre = config.preprocessing
print(pre)

## 1. Güvenli yükleme ve kanal kontrolü

In [ ]:
records = scan_labeled_directory(config.paths.labeled_dir)
example = records[len(records) // 2]
raw = load_rgb_image(example.path)
print(example.path.name)
print("mode:", raw.mode, "| size (W x H):", raw.size, "| channels:", len(raw.getbands()))

## 2. Grafik alanının kırpılması

`plot_box = (left, top, right, bottom)` 2240 × 740 figür düzenindeki heat-map alanıdır (tüm görsellerde aynı
konumda olduğu doğrulandı). Farklı boyutta bir görsel gelirse kutu orantılı ölçeklenir.

In [ ]:
cropped = crop_plot_area(raw, pre)
print("cropped size (W x H):", cropped.size, "| aspect ratio: %.2f : 1" % (cropped.width / cropped.height))
plot_image_grid([np.asarray(raw), np.asarray(cropped)], ["raw figure", "cropped heat-map"], columns=1)
plt.show()

## 3. Giriş boyutu: 512 × 128 değerlendirmesi

In [ ]:
stretched_512 = cropped.resize((512, 128))
letterbox_512 = letterbox_resize(cropped, (512, 128), pre.pad_value, pre.resample)
final = preprocess_image(raw, pre)

ratio = cropped.width / cropped.height
print(f"heat-map aspect {ratio:.2f}:1 | 512x128 = 4.00:1 -> stretching distorts by {abs(ratio / 4 - 1):.0%}")
print(f"640x128 = 5.00:1 -> distortion {abs(ratio / 5 - 1):.1%} (handled by tiny letterbox padding)")
plot_image_grid(
    [np.asarray(stretched_512), np.asarray(letterbox_512), np.asarray(final)],
    ["512x128 stretched (distorted)", "512x128 letterbox (25% padding rows)", f"{pre.target_size} letterbox (chosen)"],
    columns=1,
)
plt.show()

**Karar:** varsayılan giriş boyutu **640 × 128** (W × H). Heat-map ≈ 4.97 : 1 olduğundan 512 × 128 ya en-boy
oranını bozar ya da yüksekliğin ~%20'sini dolguya harcar. 640 × 128 ile oran korunur (yalnızca ~2 px yatay dolgu).
`PreprocessingConfig.target_size` ile değiştirilebilir. Kırpma yalnızca eksen/etiket/colorbar alanını atar.
Range ve Doppler eksenlerinin fiziksel anlamı olduğundan **çevirme/döndürme augmentasyonu kullanılmaz**.

## 4. Yinelenenlerin indeksten çıkarılması ve kayıt bazlı bölme

In [ ]:
unique_records, dedup_report = deduplicate_records(records)
print(f"indexed: {len(records)} | kept: {len(unique_records)} | {dedup_report}")

assigned = split_by_source(unique_records, config.split)
assert_no_group_leakage(assigned)

summary = pd.DataFrame([
    {"split": s, "images": len(rs), "no_target": sum(r.label == 0 for r in rs),
     "target": sum(r.label == 1 for r in rs), "recordings": len({r.source_id for r in rs})}
    for s in SPLIT_NAMES for rs in [records_in_split(assigned, s)]
])
summary

In [ ]:
save_split_csv(assigned, config.paths.splits_csv, relative_to=config.paths.project_root)
print("saved:", config.paths.splits_csv.relative_to(config.paths.project_root))

Kayıtların ≥ %5'ini tek başına oluşturan çok büyük kayıtlar (ör. ~170 görsellik beyaz gürültü kaydı) her zaman
train'e konur; böylece doğrulama/test tek bir kayda baskın olmaz. Bölme `seed` ile tekrarlanabilir.

## 5. Normalizasyon istatistikleri (yalnızca train bölümü)

In [ ]:
train_paths = [r.path for r in records_in_split(assigned, "train")]
mean, std = compute_channel_stats(train_paths, pre)
save_channel_stats(config.paths.normalization_json, mean, std)
print("mean:", np.round(mean, 4), "| std:", np.round(std, 4))

## 6. Tensör dönüşümü ve öncesi/sonrası karşılaştırma

In [ ]:
pre_norm = replace(pre, mean=mean, std=std)
tensor = array_to_tensor(np.asarray(final), pre_norm)
print("tensor shape (C, H, W):", tuple(tensor.shape), "| dtype:", tensor.dtype)
print("per-channel mean after normalisation:", tensor.mean(dim=(1, 2)).numpy().round(3))
print("per-channel std  after normalisation:", tensor.std(dim=(1, 2)).numpy().round(3))

examples = [next(r for r in records_in_split(assigned, "train") if r.label == c) for c in (0, 1)]
images, titles = [], []
for r in examples:
    image = load_rgb_image(r.path)
    images += [np.asarray(image), np.asarray(preprocess_image(image, pre))]
    titles += [f"{CLASS_NAMES[r.label]} - before", f"{CLASS_NAMES[r.label]} - after {pre.target_size}"]
plot_image_grid(images, titles, columns=2)
plt.show()